In [15]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from matplotlib.patches import Arc, Circle
from matplotlib.colors import LinearSegmentedColormap
from IPython.display import HTML

# -----------------------------
# VALUES
# -----------------------------
value = 82.6
prev_year = 77.4
global_val = 75.4

main_label = "GESAMTZUFRIEDENHEIT"
sub_label = "Top2 Box"

# -----------------------------
# SETTINGS
# -----------------------------
fps = 30
anim_sec = 5
pause_sec = 1.5
anim_frames = int(fps * anim_sec)
pause_frames = int(fps * pause_sec)
total_frames = anim_frames + pause_frames

R = 1.0
arc_lw = 9
tick_outer = 0.93
tick_inner_small = 0.76
tick_inner_big = 0.70

# Sharper gradient closer to reference image
gauge_cmap = LinearSegmentedColormap.from_list(
    "custom_gauge",
    [
        (0.00, "#ff0000"),
        (0.47, "#ff0000"),
        (0.55, "#ff7a00"),
        (0.70, "#ffd200"),
        (0.80, "#b7d96c"),
        (1.00, "#45b649"),
    ]
)

# -----------------------------
# FIGURE
# -----------------------------
fig, ax = plt.subplots(figsize=(8, 4.8), dpi=140)
fig.patch.set_facecolor("white")
ax.set_facecolor("white")


def pct_to_angle(pct):
    # 0 -> 180° ; 100 -> 0°
    return 180 - (pct / 100) * 180


def draw_gradient_arc():
    segments = 220
    vals = np.linspace(0, 100, segments + 1)

    for i in range(segments):
        a1 = pct_to_angle(vals[i + 1])
        a2 = pct_to_angle(vals[i])
        color = gauge_cmap(vals[i] / 100)
        arc = Arc(
            (0, 0), 2 * R, 2 * R,
            theta1=a1, theta2=a2,
            lw=arc_lw, color=color, capstyle="butt", zorder=2
        )
        ax.add_patch(arc)


def draw_ticks():
    majors = np.arange(0, 101, 10)
    minors = np.arange(5, 100, 10)

    for v in majors:
        th = np.deg2rad(pct_to_angle(v))
        x0, y0 = tick_outer * np.cos(th), tick_outer * np.sin(th)
        x1, y1 = tick_inner_big * np.cos(th), tick_inner_big * np.sin(th)
        ax.plot([x0, x1], [y0, y1], color="#8F8F8F", lw=1.4, zorder=1)

    for v in minors:
        th = np.deg2rad(pct_to_angle(v))
        x0, y0 = tick_outer * np.cos(th), tick_outer * np.sin(th)
        x1, y1 = tick_inner_small * np.cos(th), tick_inner_small * np.sin(th)
        ax.plot([x0, x1], [y0, y1], color="#B0B0B0", lw=1.0, zorder=1)


def draw_global_marker():
    th = np.deg2rad(pct_to_angle(global_val))
    x0, y0 = 0.84 * np.cos(th), 0.84 * np.sin(th)
    x1, y1 = 1.06 * np.cos(th), 1.06 * np.sin(th)

    ax.plot([x0, x1], [y0, y1], color="#FF4D4D", lw=1.2, zorder=4)
    ax.text(
        x1 + 0.03, y1 + 0.01,
        f"Global: {str(global_val).replace('.', ',')}%",
        color="#FF4D4D", fontsize=7, fontweight="bold",
        ha="left", va="center", zorder=4
    )


def draw_needle(pct):
    angle = np.deg2rad(pct_to_angle(pct))

    # shadow
    ax.plot(
        [0.02, 0.78 * np.cos(angle) + 0.02],
        [-0.02, 0.78 * np.sin(angle) - 0.02],
        color="black", lw=4, alpha=0.10, solid_capstyle="round", zorder=3
    )

    # needle
    ax.plot(
        [0, 0.78 * np.cos(angle)],
        [0, 0.78 * np.sin(angle)],
        color="#5F666B", lw=2.0, solid_capstyle="round", zorder=5
    )

    # center knob
    ax.add_patch(Circle((0, 0), 0.035, color="black", zorder=6))
    ax.add_patch(Circle((0, 0), 0.018, color="#1C1C1C", zorder=7))


def draw_text(pct):
    pct_str = f"{pct:.1f}".replace(".", ",")
    prev_str = f"{prev_year:.1f}".replace(".", ",")

    ax.text(0, -0.12, main_label,
            ha="center", va="center",
            fontsize=10, fontweight="bold", color="#3E3E3E")

    ax.text(0, -0.25, sub_label,
            ha="center", va="center",
            fontsize=8, color="#6A6A6A")

    ax.text(0, -0.40, f"{pct_str}%",
            ha="center", va="center",
            fontsize=20, fontweight="bold", color="#4A4A4A")

    ax.text(0, -0.55, f"2023:   {prev_str}%",
            ha="center", va="center",
            fontsize=8, fontweight="bold", color="#777777")

    ax.text(-1.10, 0.04, "0%",
            ha="center", va="center",
            fontsize=11, fontweight="bold", color="#444444")

    ax.text(1.16, 0.04, "100%",
            ha="center", va="center",
            fontsize=11, fontweight="bold", color="#444444")


def draw_gauge(pct):
    ax.clear()
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_facecolor("white")

    draw_ticks()
    draw_gradient_arc()
    draw_global_marker()
    draw_needle(pct)
    draw_text(pct)

    ax.set_xlim(-1.25, 1.25)
    ax.set_ylim(-0.68, 1.12)


def animate(frame):
    if frame >= anim_frames:
        current = value
    else:
        current = value * frame / (anim_frames - 1)

    draw_gauge(current)


anim = FuncAnimation(
    fig,
    animate,
    frames=total_frames,
    interval=1000 / fps,
    blit=False
)

plt.close(fig)
HTML(anim.to_html5_video())

# Optional save:
writer = FFMpegWriter(
    fps=fps,
    codec="libx264",
    bitrate=2500,
    extra_args=["-pix_fmt", "yuv420p"]
)
anim.save("gauge_only.mp4", writer=writer, dpi=200)

In [17]:
from google.colab import files
files.download('gauge_only.mp4')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>